In [1]:
%%bash
uv pip install -q jedi
uv pip install -q llama-index-core
uv pip install -q llama-index-llms-groq
uv pip install -q llama-index-readers-file
uv pip install -q llama-index-embeddings-huggingface
uv pip install -q llama-index-embeddings-instructor

In [ ]:
import os
import gradio as gr
from google.colab import userdata

# LlamaIndex Core
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.chat_engine import ContextChatEngine
from llama_index.core.base.llms.types import ChatMessage, MessageRole
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.memory import ChatSummaryMemoryBuffer

# LlamaIndex Integrations
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. API KEY & GLOBAL SETTINGS (The "Brain")
# This ensures LlamaIndex never defaults back to OpenAI!
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

Settings.llm = Groq(model="llama-3.1-8b-instant", temperature=0.0)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# 2. THE INFRASTRUCTURE
parser = SentenceSplitter(chunk_size=512, chunk_overlap=30)
memory = ChatSummaryMemoryBuffer.from_defaults(llm=Settings.llm, token_limit=2000)
index = None # Placeholder until a file is uploaded

# 3. THE FILE UPLOAD HANDLER
def process_new_file(file, history):
    global index
    if file is None:
        return history
    try:
        documents = SimpleDirectoryReader(input_files=[file.name]).load_data()
        nodes = parser.get_nodes_from_documents(documents)
        index = VectorStoreIndex(nodes) # Now it knows to use HuggingFace!
        memory.reset()

        success_msg = f"✅ Success! I have indexed the new manuscript into {len(nodes)} chunks. What would you like to critique?"
        history.append({"role": "assistant", "content": success_msg})
    except Exception as e:
        history.append({"role": "assistant", "content": f"⚠️ Error loading file: {str(e)}"})
    return history

# 4. THE CHAT HANDLERS
def add_user_message(user_input, history):
    history.append({"role": "user", "content": user_input})
    return "", history

def bot_responds(history, top_k):
    global index
    user_input = history[-1]["content"]

    if index is None:
        history.append({"role": "assistant", "content": "⚠️ Please upload a manuscript first!"})
        return history

    try:
        retriever = index.as_retriever(similarity_top_k=top_k)
        dynamic_chat_engine = ContextChatEngine.from_defaults(
            retriever=retriever,
            llm=Settings.llm,
            memory=memory,
            prefix_messages=[
                ChatMessage(
                    role=MessageRole.SYSTEM,
                    content="You are a rigorous peer-reviewer and expert scientific assistant. Analyze the provided manuscript deeply."
                )
            ]
        )
        response = dynamic_chat_engine.chat(user_input)
        history.append({"role": "assistant", "content": str(response)})
    except Exception as e:
        history.append({"role": "assistant", "content": f"⚠️ Backend Error: {str(e)}"})
    return history

# 5. THE GRADIO UI
with gr.Blocks(title="AI Peer-Reviewer App", theme=gr.themes.Soft()) as demo_custom:
    gr.Markdown("<h1 style='text-align: center;'>🔬 AI Peer-Reviewer (Dynamic Edition)</h1>")

    with gr.Row():
        with gr.Column(scale=1, min_width=250):
            gr.Markdown("### 📄 Document Upload")
            file_upload = gr.File(label="Upload a PDF Manuscript", file_types=[".pdf"])

            gr.Markdown("### ⚙️ Engine Settings")
            top_k_slider = gr.Slider(minimum=1, maximum=15, step=1, value=5, label="Retrieval Breadth (Top-K)")
            clear_btn = gr.Button("🗑️ Clear Conversation")

        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                label="Peer-Review Assistant",
                height=500,
                type="messages",
                value=[{"role": "assistant", "content": "Hello! Upload a manuscript on the left, or ask a question about the current one."}]
            )
            msg_input = gr.Textbox(label="Your Question", placeholder="Press Enter to send...", lines=1)

            user_action = msg_input.submit(fn=add_user_message, inputs=[msg_input, chatbot], outputs=[msg_input, chatbot], queue=False)
            user_action.then(fn=bot_responds, inputs=[chatbot, top_k_slider], outputs=[chatbot])

            file_upload.upload(fn=process_new_file, inputs=[file_upload, chatbot], outputs=[chatbot])

            def clear_everything():
                memory.reset()
                return [{"role": "assistant", "content": "Memory cleared. Let's start a new review."}], None
            clear_btn.click(fn=clear_everything, inputs=None, outputs=[chatbot, file_upload])

demo_custom.launch(share=True, debug=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_5146/181939630.py:78: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Peer-Reviewer App", theme=gr.themes.Soft()) as demo_custom:
/tmp/ipykernel_5146/181939630.py:91: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bfb7e4029b03fed361.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
